In [4]:
import pickle
import os
import pandas as pd
import numpy as np

def extract_combined_metrics():
    print("=== COMBINED METRICS: Network Structure + Adoption Speed & Probability ===\n")

    # 1. Find all folders starting with 'data_'
    base_dir = 'data'
    data_folders = [
        os.path.join(base_dir, f) 
        for f in os.listdir(base_dir) 
        if os.path.isdir(os.path.join(base_dir, f)) and f.startswith('data_')
    ]    
    if not data_folders:
        print("ERROR: No folders starting with 'data_' found.")
        return

    summary_rows = []

    # 2. Loop through each folder
    for folder in data_folders:
        file_path = os.path.join(folder, "intervention_data.pkl")
        
        if not os.path.exists(file_path):
            continue

        try:
            with open(file_path, "rb") as f:
                data = pickle.load(f)

            # =========================
            # PART A: NETWORK METRICS
            # =========================
            print(f"--- SCENARIO: {folder} ---")
            
            raw_stats = data.get("network_df")
            network_summary = {}

            if raw_stats is not None:
                # Ensure it's a DataFrame
                if isinstance(raw_stats, list):
                    df_net = pd.DataFrame(raw_stats)
                else:
                    df_net = pd.DataFrame(raw_stats)

                # Calculate means for key structural metrics
                # We save these to print and to export later
                if 'avg_clustering' in df_net.columns:
                    network_summary['Avg_Clustering'] = df_net['avg_clustering'].mean()
                if 'avg_degree' in df_net.columns:
                    network_summary['Avg_Degree'] = df_net['avg_degree'].mean()
                if 'largest_component_size' in df_net.columns:
                    network_summary['Max_Component'] = df_net['largest_component_size'].mean()

                print(" [Network Structure]")
                print(f"   Avg Clustering:    {network_summary.get('Avg_Clustering', 0):.4f}")
                print(f"   Avg Degree:        {network_summary.get('Avg_Degree', 0):.2f}")
                print(f"   Largest Component: {network_summary.get('Max_Component', 0):.1f}")
            else:
                print(" [Network Structure] No network data found.")

            # ======================================
            # PART B: ADOPTION SPEED & PROBABILITY 
            # =====================================
            print(" [Adoption Dynamics]")
            
            scenarios = {
                "Baseline": data.get("baseline_X"),
                "Subsidy":  data.get("subsidy_X")
            }

            for label, trajectories in scenarios.items():
                if trajectories is None:
                    continue

                # 1. Probability of High Adoption (>= 80%)
                final_values = [traj[-1] for traj in trajectories]
                success_count = sum(x >= 0.80 for x in final_values)
                total_trials = len(trajectories)
                prob_high = success_count / total_trials

                # 2. Speed to Stability (Time step crossing 80%)
                speeds = []
                for traj in trajectories:
                    # Get indices where adoption >= 0.80
                    high_share_indices = np.where(traj >= 0.80)[0]
                    if len(high_share_indices) > 0:
                        speeds.append(high_share_indices[0]) # First crossing
                
                # Calculate Avg Speed (only for successful trials)
                if speeds:
                    avg_speed = np.mean(speeds)
                    std_speed = np.std(speeds)
                else:
                    avg_speed = np.nan
                    std_speed = 0.0

                print(f"   {label:10} -> Prob(>=80%): {prob_high*100:5.1f}% | Speed to 80%: {avg_speed:5.1f} steps")
    

            print("-" * 40)

        except Exception as e:
            print(f"Error processing {folder}: {e}")


if __name__ == "__main__":
    extract_combined_metrics()

=== COMBINED METRICS: Network Structure + Adoption Speed & Probability ===

--- SCENARIO: data\data_BA ---
 [Network Structure]
   Avg Clustering:    0.0957
   Avg Degree:        3.95
   Largest Component: 150.0
 [Adoption Dynamics]
   Baseline   -> Prob(>=80%):  49.5% | Speed to 80%:   1.5 steps
   Subsidy    -> Prob(>=80%):  76.0% | Speed to 80%:   0.8 steps
----------------------------------------
--- SCENARIO: data\data_BA_late ---
 [Network Structure]
   Avg Clustering:    0.0957
   Avg Degree:        3.95
   Largest Component: 150.0
 [Adoption Dynamics]
   Baseline   -> Prob(>=80%):  49.5% | Speed to 80%:   1.5 steps
   Subsidy    -> Prob(>=80%):  54.5% | Speed to 80%:   4.5 steps
----------------------------------------
--- SCENARIO: data\data_BA_strong_b ---
 [Network Structure]
   Avg Clustering:    0.0957
   Avg Degree:        3.95
   Largest Component: 150.0
 [Adoption Dynamics]
   Baseline   -> Prob(>=80%):  49.5% | Speed to 80%:   1.5 steps
   Subsidy    -> Prob(>=80%):  8